# **CREACIÓN DEL ENTORNO (.venv)**   
Para que funcione correctamente:
1. Instalamos python la versión 3.13.12 (es la que he probado yo)

2. Creamos un entorno virtual (recomendado para no tener trescientos paquetes instalados)
    - Haciendo Crtl+Shift+P y seleccionando "Python: Create Environment".
    - Te tiene que haber creado una dirección en la carpeta .venv.
    - Podemos comprobarlo abriendo una terminal y ponemos "pip --version" y tiene que salir la dirección de la carpeta .venv.

# **BIBLIOTECAS NECESARIAS**

In [17]:
# Para la extracción de los ids de los videos
import requests
import re
import json
from bs4 import BeautifulSoup
from wonderwords import RandomWord
import random
import datetime
from tqdm import tqdm
import time


# Para la extracción de videos con la API
from googleapiclient.discovery import build
from yt_dlp import YoutubeDL
import pprint as pprint
import re
import glob
import os

import numpy as np
import pandas as pd
from pandas import DataFrame

# **EXTRACCIÓN DE IDS**

Función para devolver 100 ids - los resultados de una query. Con esta función nos aseguramos de que tienen subtítulos y duración media.

In [16]:
def get_video_ids(query):
    '''
    Devuelve una lista de 100 video_ids de YouTube a partir de una consulta de búsqueda
    Estos vídeos cumplen con el filtro de formato de duración media y subtítulos presentes
    '''
    
    url = "https://www.youtube.com/results"
    params = {"search_query": query,
              "sp": "EgQQASgB" #Filtro para videos formato media duración con subtítulos
              }
    headers = {
        "User-Agent": "Mozilla/5.0"
    }
    response = requests.get(url, params=params, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Buscar el script que contiene ytInitialData
    scripts = soup.find_all("script")

    if not scripts:
        print("got no scripts for query", query)

    for script in scripts:
        if "ytInitialData" in script.text:
            json_text = re.search(r"ytInitialData\s*=\s*(\{.*\});", script.text)
            if json_text:
                data = json.loads(json_text.group(1))
                break
    else:
        print("got no ytInitialData for query", query)
        return []

    # Buscar todos los videoId dentro del JSON
    video_ids = set()
    def extract_ids(obj):
        if isinstance(obj, dict):
            for k, v in obj.items():
                if k == "videoId":
                    video_ids.add(v)
                else:
                    extract_ids(v)
        elif isinstance(obj, list):
            for item in obj:
                extract_ids(item)

    extract_ids(data)

    return list(video_ids)


# 🔹 Ejemplo
ids = get_video_ids("x after:2026-02-15")
print(ids)


['byf0xtLjpnI', 'kLYWqTCt6Z0', 'ux5-7W8CZxI', 'oWD6ZzSxrkk', 'eXzVgFWQKzI', 'A9XwPiM3ZoI', 'Z3ZvxIz20-c', 'UxNxNagP5Gg', '3m4oCrY4gAs', 'gTqs_P7tk2c', 'wKuQd9mAOUw', '1JmcBfoPWyY', 'nyXLwMY1QfE', 'n-bEIjHEEc8', 'J-7elFtj9Iw', 'vJcI88xvZgI', 'NcGK6ZlPYRM', 'JHbZuGVdtHw', 'gE5mFMgvRgw', 'PoD9YBZR0O4']


Función para encontrar ids de videos aleatorios (cada id corresponde a una búsqueda de una palabra aleatoria)

In [ ]:
def get_random_ids(num_ids=25, after_date=None, before_date=None):
    '''
    Devuelve una lista de "num_ids" video_ids aleatorios a partir de palabras aleatorias y la función get_video_ids.
    Por cada palabra aleatoria se realiza una búsqueda en Youtube de videos que contengan esa palabra en el título y que hayan sido publicados entre "after_date" y "before_date".
    '''
    ##Habría que añadir algo que controle que la fecha de inicio no sea posterior a la de fin, o que no se introduzcan fechas futuras, etc. 
    # Porque si no se mete en un bucle infinito.
    lista_palabras_aleatorias = []
    lista_ids_aleatorios = []
    w = RandomWord()
    while len(lista_ids_aleatorios) < num_ids:
        try: 
            random_word = w.word()
            query = f'\"{random_word}\" intitle:{random_word}'
            if after_date:
                query += f' after:' + str(after_date)
            if before_date:
                query += f' before:' + str(before_date)
            #print("running query", query)
            lista_ids_aleatorios.append(random.choice(get_video_ids(query)))
            lista_palabras_aleatorias.append(random_word)
            time.sleep(0.2) #para no hacer muchas queries seguidas
        except Exception as e: print("Ran into exception", e, "for word", random_word) #happens quite often when no videos found for a word in the last day

    return lista_palabras_aleatorias, lista_ids_aleatorios

In [19]:
palabras, ids = get_random_ids(num_ids=5, after_date=str(datetime.date.today()-datetime.timedelta(days=1)))
#print(ids)
print(list(zip(palabras,ids)))

Ran into exception Cannot choose from an empty sequence for word multimedia
Ran into exception Cannot choose from an empty sequence for word espalier
Ran into exception Cannot choose from an empty sequence for word emphasis
Ran into exception Cannot choose from an empty sequence for word morsel
[('lose', '71oZzLpb6hQ'), ('mining', 'm1sTUpqMx1k'), ('donation', 'FeSd26rltWE'), ('switching', 'jj7h8zH5LAc'), ('pot', 'ozDUUGfSnXU')]


In [20]:
#ejemplo de extraccion de informacion con requests para un video. 
# En principio este método de extracción no se va a utilizar, hacemos la extracción con la API para tener acceso a los subtítulos, etc
url = f"https://www.youtube.com/watch?v={ids[0]}"
headers = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=headers)

match = re.search(r"ytInitialPlayerResponse\s*=\s*(\{.*?\});", r.text)

data = json.loads(match.group(1))

data['videoDetails']

{'videoId': '71oZzLpb6hQ',
 'title': 'Will Maryam lose hope forever by falling in love again?',
 'lengthSeconds': '3313',
 'channelId': 'UCKj5096Sn_uc5llfC_7uFsQ',
 'isOwnerViewing': False,
 'shortDescription': 'This description shows that Maryam is at a sensitive point in her life, where new love may, instead of bringing her peace, distance her from her hope and future. This sentence creates an atmosphere of doubt, fear of repeating the past, and inner conflict, making the reader curious to know what decision Maryam will make in the face of love and hope.\n\n\n\n\n#Love\n#Hope\n#Mary\n#Love_again\n#Heart_letter\n#Relationships\n#Love_story\n#Feelings\n#Emotional_struggle\n#Hard_choice',
 'isCrawlable': True,
 'thumbnail': {'thumbnails': [{'url': 'https://i.ytimg.com/vi/71oZzLpb6hQ/hqdefault.jpg?sqp=-oaymwEiCKgBEF5IWvKriqkDFQgBFQAAAAAYASUAAMhCPQCAokN4AQ==&rs=AOn4CLBH-NIkCN_79mYub170f-Pgu_wGlQ',
    'width': 168,
    'height': 94},
   {'url': 'https://i.ytimg.com/vi/71oZzLpb6hQ/hqdefaul

# **GUARDAR API KEY Y CREAR EL "SERVIDOR"**
La función build nos construye un objeto de la API de Youtube, que usaremos para que llame a la API y acceder a los datos.

In [21]:
# Guardamos nuestra API_KEY leyendo de la carpeta privada
with open("./Private/claves.json", "r", encoding="utf-8") as archivo:
    claves = json.load(archivo)
API_KEY = claves["Clave_API"]
if API_KEY:
    print("Loaded API_KEY succesfully")

Loaded API_KEY succesfully


In [22]:
# Construimos el objeto de la API de YouTube
youtube = build('youtube', 'v3', developerKey=API_KEY)

# **FUNCIÓN PARA LIMPIAR EL TEXTO**

In [23]:
#Función temporal para limpiar texto
import re

def clean_vtt(text):
    text = re.sub(r"WEBVTT.*\n", "", text)
    text = re.sub(r"\d+:\d+:\d+\.\d+ --> .*", "", text)
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"\n+", "\n", text)
    return text.strip()

def clean_vtt_smart(text):
    lines = []
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        if not lines or not line in lines[-1]:
            lines.append(line)
    return " ".join(lines)

# **GENERAMOS UNA PEQUEÑA BASE DE DATOS**
Lo guardaremos en un archivo csv, guardando la información de los 5 videos que hemos encontrado antes.


In [27]:
# Lista de los id de los videos (usando el id el coste de la consulta es el mínimo)
list_id = ids
print("ids", list_id)

# Creo el dataframe vacío
df_data = pd.DataFrame({
    "ID": [],
    "Titulo": [],
    "Descripcion": [],
    "Visualizaciones": [],
    "Numero_likes": [],
    "Duracion": [],
    "Fecha_publicacion": [],
    "Titulo_canal": [],
    "Subtitulos": []
})
df_data

ids ['71oZzLpb6hQ', 'm1sTUpqMx1k', 'FeSd26rltWE', 'jj7h8zH5LAc', 'ozDUUGfSnXU']


,ID,Titulo,Descripcion,Visualizaciones,Numero_likes,Duracion,Fecha_publicacion,Titulo_canal,Subtitulos


In [33]:
from urllib import response


def get_info(id_video):
    """ Saca la información del video, si hay subtítulos los limpia, y añade dichos datos al dataframe. """
    
    # Inicializamos la variable de los subtitulos
    cleant_sub = None

    # Hacemos la llamada a la API para obtener los detalles del video
    request = youtube.videos().list(
        part="snippet,contentDetails,statistics,status,topicDetails,recordingDetails",
        id=id_video
    )  

    # Ejecutamos la solicitud
    response = request.execute()

    if not response["items"]:
        return None
    
    video = response["items"][0]

    has_captions = video["contentDetails"].get("caption") in ["true", True]

    # --- SUBTÍTULOS ---
    if (video["contentDetails"].get("caption") == "true"):
        url =  "https://www.youtube.com/watch?v=" + id_video
        # Opciones de descarga
        ydl_opts = {
            "skip_download": True,
            "writesubtitles": True,
            "writeautomaticsub": True,      # subtítulos automáticos
            "subtitleslangs": ["en"], # idioma
            "subtitlesformat": "vtt",       # formato
            "outtmpl": f"subs/{id_video}.%(ext)s",
            "quiet": True
        }

        with YoutubeDL(ydl_opts) as ydl:
            ydl.download([url])

        vtt_file = f"subs/{id_video}.en.vtt"

        if os.path.exists(vtt_file):
            with open(vtt_file, "r", encoding="utf-8") as f:
                subtitles = f.read()

            clean_sub = clean_vtt(subtitles)
            cleant_sub = clean_vtt_smart(clean_sub)

            # Borramos el archivo de subtítulos descargado tras limpiarlo
            # os.remove(vtt_file)
    
    # --- GENEROS ---
    generos = video.get("topicDetails", {}).get("topicCategories", [])

    if generos:
        generos = [
            genre.split("/")[-1].replace("_", " ")
            for genre in generos
        ]
        generos_str = ", ".join(generos)
    
    else:
        generos_str = "None"

    # --- DATAFRAME ---
    df_video = pd.DataFrame({
        "ID": id_video,
        "Titulo": video["snippet"]["title"],
        "Descripcion": video["snippet"]["description"],
        "Visualizaciones": video["statistics"]["viewCount"],
        "Numero_likes": video["statistics"].get("likeCount", None),
        "Duracion": video["contentDetails"]["duration"],
        "Fecha_publicacion": video["snippet"]["publishedAt"],
        "Titulo_canal": video["snippet"]["channelTitle"],
        "Subtitulos": cleant_sub if has_captions and cleant_sub else "None",
        "Generos": generos_str,
        "Made for kids": video["status"]["madeForKids"]
    }, index=[0])

    return df_video

In [34]:
df_videos = []

for id in tqdm(list_id):
    try:
        df_videos.append(get_info(id))
        #print(df_videos)
        time.sleep(0.5) #to not get too many requests error
    except Exception as e: print("Ran into exception", e, "for video", id) #for some videos the downloader does not work for some reason

df_data = pd.concat(df_videos, ignore_index=True)

 20%|██        | 1/5 [00:01<00:06,  1.67s/it]WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one
ERROR: Unable to download video subtitles for 'en': HTTP Error 429: Too Many Requests
 40%|████      | 2/5 [00:03<00:05,  1.82s/it]

Ran into exception ERROR: Unable to download video subtitles for 'en': HTTP Error 429: Too Many Requests for video m1sTUpqMx1k


 60%|██████    | 3/5 [00:05<00:03,  1.71s/it]WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one


 80%|████████  | 4/5 [00:06<00:01,  1.72s/it]WARNING: [youtube] No supported JavaScript runtime could be found. Only deno is enabled by default; to use another runtime add  --js-runtimes RUNTIME[:PATH]  to your command/config. YouTube extraction without a JS runtime has been deprecated, and some formats may be missing. See  https://github.com/yt-dlp/yt-dlp/wiki/EJS  for details on installing one


100%|██████████| 5/5 [00:08<00:00,  1.79s/it]


In [35]:
df_data

,ID,Titulo,Descripcion,Visualizaciones,Numero_likes,Duracion,Fecha_publicacion,Titulo_canal,Subtitulos,Generos,Made for kids
0,71oZzLpb6hQ,Will Maryam lose hope forever by falling in lo...,This description shows that Maryam is at a sen...,18088,1111,PT55M14S,2026-02-20T11:30:00Z,MARYAM,None,"Food, Lifestyle (sociology)",False
1,FeSd26rltWE,Quadrant2Design Exhibition Stand Donation to T...,"Last month, we had the absolute honour of welc...",46,0,PT6M47S,2026-02-20T14:21:55Z,Quadrant2Design,"Kind: captions Language: en The pictures, the ...","Hobby, Lifestyle (sociology)",False
2,jj7h8zH5LAc,Switching Contact Plates on the Bullet Blender...,https://www.nextadvance.com/bullet-blender-hom...,4,0,PT1M53S,2026-02-20T21:09:14Z,Next Advance,"Kind: captions Language: en Hi, my name is Nic...",None,False
3,ozDUUGfSnXU,Deep Dish Chicken Pot Pie with Sauce Supreme,This is American comfort food built on classic...,1,0,PT2M20S,2026-02-20T16:01:20Z,The Good Plate,Kind: captions Language: en Good afternoon and...,"Food, Lifestyle (sociology)",False


Una vez que tenemos el dataframe, lo exportamos en formato csv.

In [32]:
data_csv = df_data.to_csv("data_videos.csv", index=False)

# **UNA SOLA FUNCIÓN PARA TODA LA EXTRACCIÓN DE DATOS**

In [36]:
# Guardamos nuestra API_KEY leyendo de la carpeta privada
with open("./Private/claves.json", "r", encoding="utf-8") as archivo:
    claves = json.load(archivo)
API_KEY = claves["Clave_API"]
if API_KEY:
    print("Loaded API_KEY succesfully")

# Construimos el objeto de la API de YouTube
youtube = build('youtube', 'v3', developerKey=API_KEY)

Loaded API_KEY succesfully


In [41]:
def collect_all_data(num_videos):
    palabras, ids = get_random_ids(num_ids=num_videos, after_date=str(datetime.date.today()-datetime.timedelta(days=1)))
    df_videos = []
    for id in ids: #tqdm
        try:
            df_videos.append(get_info(id))
            #print(df_videos)
            time.sleep(0.5) #to not get too many requests error
        except Exception as e: print("Ran into exception", e, "for video", id) #for some videos the downloader does not work for some reason

    df_data = pd.concat(df_videos, ignore_index=True)
    data_csv = df_data.to_csv("data_videos.csv", index=False)
    return df_data

In [42]:
collect_all_data(5)

Ran into exception Cannot choose from an empty sequence for word undergo
Ran into exception Cannot choose from an empty sequence for word raincoat
Ran into exception Cannot choose from an empty sequence for word custard
Ran into exception Cannot choose from an empty sequence for word intentionality


ERROR: Unable to download video subtitles for 'en': HTTP Error 429: Too Many Requests


Ran into exception ERROR: Unable to download video subtitles for 'en': HTTP Error 429: Too Many Requests for video 5yi7_SSVDZU


,ID,Titulo,Descripcion,Visualizaciones,Numero_likes,Duracion,Fecha_publicacion,Titulo_canal,Subtitulos,Generos,Made for kids
0,freBsct5MjY,A Public School Teacher's Honest Take on Homes...,Want to be alerted when new episodes drop? Joi...,66,7,PT45M22S,2026-02-20T14:01:06Z,Ex-Homeschoolers Club,Kind: captions Language: en before I got my te...,Society,False
1,xgpZGHEGHXQ,Unexpected Connections in Crooked Soley: Allan...,This is a video about Unexpected Connections i...,61,1,PT37S,2026-02-20T18:01:10Z,Curious Dimension,"Kind: captions Language: en They said, ""Oh, th...",None,False
2,ejGbrylEcHs,2026 02 CEFT 08 IA défis de modélisation et de...,https://www.youtube.com/playlist?list=PLL6sFPM...,18,0,PT24M30S,2026-02-20T22:31:41Z,3AF groupe Ile de France,Kind: captions Language: en Let's start from t...,Knowledge,False
3,ak-zzMe7nng,Pave My Way 🧱 | Cinematic Motivational Trap / ...,Flowbotik – Pave My Way [Official Audio] \nGen...,8,2,PT3M7S,2026-02-20T18:00:03Z,FLOWBOTIK,Kind: captions Language: en They say wait your...,"Electronic music, Hip hop music, Music",False
